# Train the sign model

This notebook reads `results/landmarks.csv`, trains a RandomForest model, checks two types of train/test split, shows common mistakes, and saves the final model.

The model uses only the signs that were kept during landmark extraction.

## What the two data sets mean

The **train set** teaches the model. The **test set** checks the model on examples it did not use for learning.

The careful split keeps the first 70% of each sign for training and the last 30% for testing. This helps reduce leakage from nearby pictures.

The random split is included only for comparison. Its score can be too high when similar pictures of the same person appear on both sides.

In [2]:
from pathlib import Path
import csv
from collections import Counter
import joblib
import numpy as np
import matplotlib
matplotlib.use("Agg")  # save figures to files
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import train_test_split

PROJECT = Path.cwd()
if PROJECT.name == "step3_train":
    PROJECT = PROJECT.parent
FEATURES = [f"f{i}" for i in range(129)]
print("Project:", PROJECT)

Project: c:\Users\User\Documents\tunisian sign language to speech


## 1. Load the landmark table

The landmark notebook creates this file. Each row is one picture. The `sign` column is the answer, and `f0` to `f128` are the 129 numbers used as features.

In [3]:
# Load the table made by extract_landmarks.py without Pandas.
landmarks_path = PROJECT / "results" / "landmarks.csv"
rows = []
with landmarks_path.open(newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        rows.append(row)

if not rows:
    raise SystemExit(f"No landmark rows found in {landmarks_path}")

# Keep the same sign and path information, with the 129 features as NumPy numbers.
rows.sort(key=lambda row: (row["sign"], row["path"]))
y = np.array([row["sign"] for row in rows])
paths = np.array([row["path"] for row in rows])
X = np.array([[float(row[name]) for name in FEATURES] for row in rows], dtype=float)

print("Pictures per sign:")
for sign, count in sorted(Counter(y).items()):
    print(f"{sign}: {count}")

Pictures per sign:
3aslema: 51
baladya: 109
banka: 63
dar: 73
jom3a: 56
labes: 21
louage: 30
lyoum: 49
non: 66
om: 46
oui: 89
sbitar: 46
taxi: 96


## 2. Make the careful train/test split

Pictures that are next to each other often come from the same person. Sorting by sign and file name, then taking the first 70% for training and the last 30% for testing, helps keep similar pictures from landing on both sides.

In [4]:
# Split A (careful): first 70% of each sign to train, last 30% to test.
train_idx, test_idx = [], []
for sign in sorted(set(y)):
    group_idx = [i for i, row_sign in enumerate(y) if row_sign == sign]
    cut = int(len(group_idx) * 0.7)
    train_idx += group_idx[:cut]
    test_idx += group_idx[cut:]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

model = RandomForestClassifier(
    n_estimators=200,
    random_state=0,
    class_weight="balanced",
)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("Accuracy with the careful split:", round(accuracy_score(y_test, pred) * 100, 1), "%")
print(classification_report(y_test, pred))

Accuracy with the careful split: 93.4 %
              precision    recall  f1-score   support

     3aslema       1.00      1.00      1.00        16
     baladya       0.97      0.91      0.94        33
       banka       0.90      0.95      0.92        19
         dar       1.00      0.91      0.95        22
       jom3a       1.00      1.00      1.00        17
       labes       1.00      1.00      1.00         7
      louage       0.82      1.00      0.90         9
       lyoum       1.00      0.93      0.97        15
         non       0.95      0.95      0.95        20
          om       1.00      1.00      1.00        14
         oui       0.79      0.96      0.87        27
      sbitar       0.70      0.50      0.58        14
        taxi       1.00      1.00      1.00        29

    accuracy                           0.93       242
   macro avg       0.93      0.93      0.93       242
weighted avg       0.94      0.93      0.93       242



## 3. Compare with a random split

This split is useful as a warning. If its score is much higher, similar pictures may be leaking between the train and test sets.

In [5]:
# Split B (comparison): a random split.
Xtr, Xte, ytr, yte = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=0,
    stratify=y,
)
rand_model = RandomForestClassifier(
    n_estimators=200,
    random_state=0,
    class_weight="balanced",
).fit(Xtr, ytr)
print("Accuracy with a random split:   ", round(rand_model.score(Xte, yte) * 100, 1), "%")
print("(If this is much higher, similar pictures leak between train and test.)")

Accuracy with a random split:    95.4 %
(If this is much higher, similar pictures leak between train and test.)


## 4. Check the signs the model confuses

A confusion matrix shows which real signs are predicted as other signs. This helps us find signs that look alike.

In [6]:
# Find the most common mistakes.
labels = sorted(set(y))
cm = confusion_matrix(y_test, pred, labels=labels)
pairs = [
    (cm[i, j], labels[i], labels[j])
    for i in range(len(labels))
    for j in range(len(labels))
    if i != j and cm[i, j] > 0
]
print("Most common mistakes:")
for n, real, guess in sorted(pairs, reverse=True)[:5]:
    print(f"  {real} was predicted as {guess}: {n} times")

fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    pred,
    labels=labels,
    normalize="true",
    values_format=".2f",
    xticks_rotation=90,
    ax=ax,
    colorbar=False,
)
fig.tight_layout()
fig.savefig(PROJECT / "results" / "confusion_matrix.png")
plt.close(fig)
print("Confusion matrix saved in results/confusion_matrix.png")

Most common mistakes:
  sbitar was predicted as oui: 6 times
  baladya was predicted as sbitar: 3 times
  sbitar was predicted as baladya: 1 times
  oui was predicted as non: 1 times
  non was predicted as oui: 1 times
Confusion matrix saved in results/confusion_matrix.png


## 5. Train the final model

After checking the model, train one final model with all available landmark rows. Save it for the gadget step.

In [7]:
# Train again on all the data and save the model.
final = RandomForestClassifier(
    n_estimators=200,
    random_state=0,
    class_weight="balanced",
)
final.fit(X, y)
model_path = PROJECT / "step3_train" / "model.joblib"
joblib.dump(final, model_path)
print("Model saved in", model_path)

Model saved in c:\Users\User\Documents\tunisian sign language to speech\step3_train\model.joblib
